In [2]:
#Ensure the module can be successfully import
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()

while (
    not (PROJECT_ROOT / "src").exists()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Current directory:", Path.cwd())

Project root: /Users/shingshing/MyDocuments/HKU/Sem3/DASE7310/Hong-Kong-Equity-Forecasting-Portfolio
Current directory: /Users/shingshing/MyDocuments/HKU/Sem3/DASE7310/Hong-Kong-Equity-Forecasting-Portfolio


In [4]:
import pandas as pd

from src.hk_equity.data.preprocess import (
    calculate_returns,
    to_weekly_close,
)
from src.hk_equity.features.weekly_features import (
    WeeklyFeatureSpec,
    build_live_features,
    build_weekly_training_features,
    drop_incomplete_feature_rows,
    get_feature_columns,
)

daily_close = pd.read_csv(
    "data/raw/latest_daily_close.csv",
    index_col="Date",
    parse_dates=True,
).sort_index()

portfolio_tickers = [
    "9988.HK",
    "0066.HK",
    "1398.HK",
    "3988.HK",
    "0981.HK",
    "0005.HK",
    "1810.HK",
    "0016.HK",
    "1113.HK",
    "0700.HK",
]

portfolio_daily_close = daily_close[portfolio_tickers]
hsi_daily_close = daily_close["^HSI"].dropna().to_frame()

portfolio_weekly_close = to_weekly_close(
    portfolio_daily_close,
    weekly_frequency="W-FRI",
)

hsi_weekly_close = to_weekly_close(
    hsi_daily_close,
    weekly_frequency="W-FRI",
)

portfolio_weekly_returns = calculate_returns(
    portfolio_weekly_close,
).dropna(how="all")

hsi_weekly_returns = calculate_returns(
    hsi_weekly_close,
).iloc[:, 0].dropna()

spec = WeeklyFeatureSpec(
    include_ratio_features=False,
    include_cross_sectional_features=True,
    winsorize_features=False,
)

training_features = build_weekly_training_features(
    weekly_returns=portfolio_weekly_returns,
    benchmark_returns=hsi_weekly_returns,
    spec=spec,
)

feature_columns = get_feature_columns(training_features)

clean_training_features = drop_incomplete_feature_rows(
    feature_table=training_features,
    feature_columns=feature_columns,
)

live_features = build_live_features(
    weekly_returns=portfolio_weekly_returns,
    benchmark_returns=hsi_weekly_returns,
    spec=spec,
)

print("Training shape:", training_features.shape)
print("Clean training shape:", clean_training_features.shape)
print("Live shape:", live_features.shape)
print("Number of tickers:", training_features["ticker"].nunique())
print("Feature count:", len(feature_columns))
print(training_features.columns.tolist())

KeyError: '^HSI'

In [ ]:
assert "^HSI" in daily_close.columns
assert training_features["ticker"].nunique() == 10
assert live_features["ticker"].nunique() == 10
assert "market_return_lag_1" in training_features.columns
assert "excess_return_lag_1" in training_features.columns
assert "return_lag_52" in training_features.columns
assert "rolling_mean_26" in training_features.columns
assert "downside_deviation_4" in training_features.columns
assert "momentum_spread_4_12" in training_features.columns